In [0]:
%pip install xgboost

In [0]:
import mlflow
import xgboost as xgb
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from pyspark.sql.functions import pandas_udf
from delta.tables import DeltaTable

mlflow.set_registry_uri("databricks-uc")

In [0]:
import json

MODEL_URI = "models:/stocks.models.combined_predictor@champion"

client = mlflow.MlflowClient()
model_version = client.get_model_version_by_alias("stocks.models.combined_predictor", "champion")
run = client.get_run(model_version.run_id)

feature_cols = json.loads(run.data.params["feature_cols"])
print(f"Model version : {model_version.version}")
print(f"Feature count : {len(feature_cols)}")

In [0]:
# Serialize booster to bytes so workers can reconstruct it without network calls
wrapper = mlflow.xgboost.load_model(MODEL_URI)
booster_bytes = wrapper.get_booster().save_raw()

@pandas_udf(DoubleType())
def predict_proba(*feature_series: pd.Series) -> pd.Series:
    booster = xgb.Booster()
    booster.load_model(bytearray(booster_bytes))
    features_df = pd.concat(feature_series, axis=1)
    features_df.columns = feature_cols
    return pd.Series(booster.predict(xgb.DMatrix(features_df)))

In [0]:
df = (
    spark.table("stocks.gold.stocks_combined_features")
    .filter("Date >= '2026-01-01'")
    .dropna(subset=feature_cols)   # skip rows where FX/macro not available yet
)

scored = (
    df
    .withColumn("certainty",  predict_proba(*[F.col(c) for c in feature_cols]))
    .withColumn("prediction", (F.col("certainty") > 0.5).cast("integer"))
)

final_df = scored.select("Date", "company", "prediction", "certainty", "label")
display(final_df.orderBy("Date", "company"))

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS stocks.reporting")

tbl = "stocks.reporting.predictions_combined"
if not spark.catalog.tableExists(tbl):
    final_df.write.mode("overwrite").format("delta").saveAsTable(tbl)

DeltaTable.forName(spark, tbl).alias("t").merge(
    final_df.alias("s"),
    "s.Date = t.Date AND s.company = t.company"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

# Back-fill actual labels from gold table for past predictions
gold = spark.table("stocks.gold.stocks_combined_features").select("Date", "company", "label")

DeltaTable.forName(spark, tbl).alias("t").merge(
    gold.alias("g"),
    "g.Date = t.Date AND g.company = t.company"
).whenMatchedUpdate(set={"label": "g.label"}).execute()

print(f"{tbl}: {spark.table(tbl).count()} rows")
spark.table(tbl).orderBy(F.desc("Date"), "company").show(20)